<a href="https://colab.research.google.com/github/junsu122/AI_Application/blob/main/%EC%9D%B4%EB%AF%B8%EC%A7%80%EC%BA%A1%EC%85%94%EB%8B%9D/image_captioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 입문용 이미지 캡셔닝(Image Captioning) 실습 노트북


- 데이터셋: GitHub에서 제공하는 **Flickr8k** 데이터셋 (일부 샘플만 사용)
- 인코더(Encoder): 사전 학습된 **ResNet-18(CNN)**
- 디코더(Decoder): **LSTM 기반 문장 생성기**

In [1]:
import os  # 운영체제 기능(폴더 생성, 경로 처리 등)을 사용하기 위한 모듈
import re  # 정규표현식(텍스트 전처리)에 사용되는 모듈
import zipfile  # zip 파일(압축 파일)을 풀기 위해 사용하는 모듈
import random  # 무작위 샘플 추출, 시드 고정 등에 사용하는 모듈
from collections import Counter  # 단어 빈도수를 세기 위해 사용하는 자료구조

import urllib.request  # 인터넷에서 파일을 다운로드하기 위한 표준 라이브러리 모듈

import numpy as np  # 숫자 계산과 배열 연산을 편리하게 해주는 라이브러리
from PIL import Image  # 이미지 파일을 열고 다루기 위한 라이브러리(Pillow)
import matplotlib.pyplot as plt  # 그래프나 이미지를 화면에 출력하기 위한 라이브러리

import torch  # PyTorch 딥러닝 프레임워크의 핵심 패키지
from torch import nn  # 신경망 레이어를 만들기 위한 모듈
from torch.utils.data import Dataset, DataLoader  # 데이터셋과 배치 생성을 도와주는 클래스들

from torchvision import transforms  # 이미지 전처리(리사이즈, 텐서 변환 등)를 위한 모듈
from torchvision.models import resnet18, ResNet18_Weights  # 사전 학습된 ResNet-18 모델과 그 가중치 설정

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU가 있으면 GPU, 없으면 CPU를 사용하도록 설정
print("사용 중인 디바이스:", device)  # 현재 사용 중인 디바이스를 출력하여 확인

사용 중인 디바이스: cuda


In [2]:
# 재현성 위해서 seed 값 설정
def set_seed(seed: int = 42):  # seed 값을 받아서 여러 라이브러리의 난수 발생기를 고정하는 함수 정의
    random.seed(seed)  # 파이썬 기본 random 모듈의 시드를 고정
    np.random.seed(seed)  # 넘파이의 난수 시드를 고정
    torch.manual_seed(seed)  # PyTorch CPU 난수 시드를 고정
    if torch.cuda.is_available():  # 만약 GPU(CUDA)가 사용 가능하다면
        torch.cuda.manual_seed_all(seed)  # 모든 GPU의 난수 시드를 고정

set_seed(42)  # 위에서 정의한 함수를 호출하여 시드를 42로 고정


In [3]:
# 데이터 다운로드

import os
import zipfile
import urllib.request

# 전체 데이터셋이 들어갈 기본 폴더
data_dir = "./flickr8k"
os.makedirs(data_dir, exist_ok=True)

# 정상 동작하는 Flickr8k 공식 미러 URL (Jason Brownlee GitHub)
images_zip_url = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip"
text_zip_url   = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip"

images_zip_path = os.path.join(data_dir, "Flickr8k_Dataset.zip")
text_zip_path   = os.path.join(data_dir, "Flickr8k_text.zip")

def download_if_not_exists(url, save_path):
    """파일이 없을 때만 다운로드"""
    if not os.path.exists(save_path):
        print(f"다운로드 중: {url}")
        urllib.request.urlretrieve(url, save_path)
        print(f"완료: {save_path}")
    else:
        print(f"이미 존재: {save_path}")

def unzip_if_needed(zip_path, extract_to):
    """압축이 아직 안 풀려 있으면 압축 해제"""
    if not os.path.exists(extract_to):
        print(f"압축 해제 중: {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_to)
        print(f"압축 해제 완료: {extract_to}")
    else:
        print(f"이미 압축 해제됨: {extract_to}")

# 1) zip 파일 다운로드
download_if_not_exists(images_zip_url, images_zip_path)
download_if_not_exists(text_zip_url,   text_zip_path)

import zipfile
import os

zip_path = "/content/flickr8k/Flickr8k_Dataset.zip"
extract_path = "/content/flickr8k/"

print("압축 해제 중...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_path)

print("완료!")
print("압축 해제 후 폴더 목록:", os.listdir(extract_path))

다운로드 중: https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
완료: ./flickr8k/Flickr8k_Dataset.zip
다운로드 중: https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip
완료: ./flickr8k/Flickr8k_text.zip
압축 해제 중...
완료!
압축 해제 후 폴더 목록: ['Flickr8k_Dataset.zip', 'Flicker8k_Dataset', 'Flickr8k_text.zip', '__MACOSX']


## 4. 캡션 파일 로드 및 구조 이해

`Flickr8k.token.txt` 파일에는 **이미지 파일 이름과 그 이미지에 대한 여러 문장(캡션)** 이 함께 들어 있습니다.

예시 형식:

```text
1000268201_693b08cb0e.jpg#0\tA child in a pink dress is climbing up a set of stairs in an entry way .
```

- `1000268201_693b08cb0e.jpg` : 이미지 파일 이름
- `#0` : 이 이미지에 대한 0번째 캡션 (한 이미지당 5개의 캡션)
- 그 뒤 : 실제 문장

In [ ]:
import zipfile

# 데이터셋 압축 파일 경로 (이미지에서 보인 경로를 기반으로 예시)
# 'data_dir'이 'flickr8k'의 상위 폴더를 가리킨다고 가정합니다.
zip_path = os.path.join(data_dir, "Flickr8k_text.zip")

# 압축을 풀 디렉토리
extract_to_dir = data_dir

# 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_dir)
    print(f"'{zip_path}' 파일이 '{extract_to_dir}'에 성공적으로 압축 해제되었습니다.")

'./flickr8k/Flickr8k_text.zip' 파일이 './flickr8k'에 성공적으로 압축 해제되었습니다.


In [ ]:
import os

# 예시: data_dir이 'flickr8k'의 상위 디렉토리라고 가정
# 사용자 환경에 맞춰 'data_dir' 변수 설정이 필요합니다.
# 만약 'data_dir'이 이미 'flickr8k' 폴더를 가리킨다면, 아래 코드는 필요 없습니다.
# 여기서는 'flickr8k' 폴더 안에 데이터가 있다고 가정하고 경로를 설정합니다.
# **!!! 중요: 실제 환경에 맞게 data_dir 값을 설정해야 합니다 !!!**
# 예를 들어 data_dir이 '/content/' 이고 'flickr8k' 폴더가 그 안에 있다면:
# data_dir = '/content/flickr8k'
# 이 예시에서는 기존 코드의 'data_dir'이 'flickr8k' 폴더의 경로라고 가정하고 진행합니다.

# --- 4. 캡션 파일 로드 ---
# 'Flickr8k_text.zip' 압축을 푼 후, 'Flickr8k.token.txt' 파일의 실제 경로를 확인하여 수정해야 합니다.
# 일반적으로 'data_dir' 안에 압축을 풀면 'Flickr8k.token.txt' 파일이 바로 생깁니다.
# 만약 에러가 발생했다면, 파일 이름이 잘못되었거나 경로가 잘못되었을 가능성이 큽니다.

# 1. 파일 이름이 잘못되었을 경우 (혹시나 하여 오타 수정 가능성 포함):
# captions_file = os.path.join(data_dir, "Flickr8k.token.txt") # 기존 코드

# 2. 파일이 'flickr8k' 폴더 안에 있고, data_dir이 'flickr8k'의 상위 폴더인 경우:
# data_dir이 'flickr8k' 폴더를 포함하는 상위 경로일 경우 아래처럼 수정해야 합니다.
# captions_file = os.path.join(data_dir, "flickr8k", "Flickr8k.token.txt")
# (이 경우 'data_dir'의 정확한 정의가 필요합니다.)

# ***가장 일반적인 해결책: 파일 이름이 정확하다면, 압축을 풀지 않았거나 파일의 실제 위치가 다른 것입니다.***

# **Flickr8k.token.txt 파일을 찾을 수 있는 올바른 경로로 수정하세요.**
# (예시: data_dir이 데이터를 담고 있는 최상위 폴더이고, 그 안에 'Flickr8k.token.txt'가 있다고 가정합니다.)

captions_file = os.path.join(data_dir, "Flickr8k.token.txt") # 압축을 푼 파일 경로를 지정

print("캡션 파일 경로:", captions_file)

try:
    with open(captions_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("전체 캡션 라인 수:", len(lines))
    print("앞에서 3줄만 미리 보기:")
    for i in range(3):
        print(lines[i].strip())

except Exception as e:
    print(f"\n파일을 읽는 중 예기치 않은 에러 발생: {e}")

캡션 파일 경로: ./flickr8k/Flickr8k.token.txt
전체 캡션 라인 수: 40460
앞에서 3줄만 미리 보기:
1000268201_693b08cb0e.jpg#0	A child in a pink dress is climbing up a set of stairs in an entry way .
1000268201_693b08cb0e.jpg#1	A girl going into a wooden building .
1000268201_693b08cb0e.jpg#2	A little girl climbing into a wooden playhouse .


## 5. 텍스트 전처리 및 이미지-캡션 매핑 만들기

이미지 파일 이름별로 여러 개의 캡션 문장을 모아 두기 위해, 다음과 같은 과정을 거칩니다.

1. 한 줄씩 읽어 **이미지 이름**과 **문장** 부분을 분리합니다.
2. 문장 안의 불필요한 기호(쉼표, 마침표 등)를 제거하고, 모두 소문자로 바꿉니다.
3. 이미지 이름을 key로 하고, 그 이미지에 대한 여러 캡션 리스트를 value로 갖는 딕셔너리를 만듭니다.

In [ ]:
# def clean_sentence(sentence: str) -> str :
#     sentence = sentence.lower()
#     sentence = re.sub(r'[^a-z ]','',sentence)
#     sentence = re.sub(r'\s+','',sentence).strip()
#     return sentence

In [ ]:
# 텍스트 전처리 및 이미지-캡션 딕셔너리 생성
def clean_sentence(sentence: str) -> str:
    sentence = sentence.lower()  # 모든 문자를 소문자 변환
    sentence = re.sub(r'[^a-z ]','', sentence)  # 알파벳 소문자와 공백 제외한 문제 제거('')
    sentence = re.sub(r'\s+',' ', sentence).strip() # 여러 개 공백을 하나로 줄이고, 양끝 공백 제거
    return sentence  # 전처리가 끝난 문장 반환

captions_dict = {}
# 이미지 파일 이름: key, 해당 이미지 문장 리스트를 value 저장할 딕셔너리 {key:value}

for line in lines:
    line = line.strip()
    if len(line) == 0:  # 빈 줄이라면
        continue        # 다음 줄로 넘어가
    image_and_caption = line.split('\t') # tab 문자 기준, 이미지 정보와 문자 분리
    if len(image_and_caption) !=2:
        continue
    # 탭으로 나눈 결과가 2가 아니라면(어라? 이미지, 캡션 하나씩 있어야 하는데)
    # 즉, 형식이 이상하면 건너 띄어
    image_id_raw, caption_raw = image_and_caption # (이미지 번호, 캡션 문장)
    image_filename = image_id_raw.split('#')[0]   # 파일이름#번호

    cleaned = clean_sentence(caption_raw)

    if len(cleaned.split()) < 3:  # 단어 수가 너무 적은 문장 (도움 안됨) >> 제거
        continue
    captions_dict.setdefault(image_filename, []).append(cleaned)
    # 해당 이미지 파일에 문장 추가

print('이미지 개수(캡션 포함):', len(captions_dict))
# 캡션이 있는 이미지 몇 개니?


이미지 개수(캡션 포함): 8092


In [ ]:
#  한 이미지에 어떤 캡션 들어 있는지 예시로 하나만 출력
# captions_dict.keys() # 파일이름

sample_key = next(iter(captions_dict.keys()))
print(sample_key)

for c in captions_dict[sample_key]:
    print('-', c)



1000268201_693b08cb0e.jpg
- a child in a pink dress is climbing up a set of stairs in an entry way
- a girl going into a wooden building
- a little girl climbing into a wooden playhouse
- a little girl climbing the stairs to her playhouse
- a little girl in a pink dress going into a wooden cabin


In [ ]:
# 입문용: subset 사용 (실제 8,092장 >> 우리는 200장 사용(random))

all_images_filenames = list(captions_dict.keys())
len(all_images_filenames)

8092

In [ ]:
subset_size = 200
if len(all_images_filenames) < subset_size:
    subset_size = len(all_images_filenames)

small_image_filenames = random.sample(all_images_filenames, subset_size)
len(small_image_filenames)

200

## 7. 단어 사전(vocabulary) 만들기

이미지 캡셔닝에서 문장을 다루려면, **단어를 숫자(index)** 로 바꾸는 과정이 필요합니다.

- 특별 토큰(special token)
  - `<pad>` : 빈 자리를 채울 때 사용하는 토큰 (길이를 맞추기 위함)
  - `<start>` : 문장이 시작됨을 알리는 토큰
  - `<end>` : 문장이 끝났음을 알리는 토큰
  - `<unk>` : 사전에 없는 단어를 대신하는 토큰

단어 빈도가 너무 낮은 단어는 모두 `<unk>`로 처리해, 사전의 크기를 적당히 줄입니다.


In [ ]:
# 단어 사전 구성
special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]

word_counter = Counter() # 각 단어가 몇 번 등장했는지 세려고

for img in small_image_filenames:
    for cap in captions_dict[img]: # 각 이미지에 대해 caption들을 순회
        for w in cap.split():      # caption에 있는 (문장의) 단어를 공백기준으로 나누어
            word_counter[w] += 1   # 해당 단어 등장 빈도 1 증가시킴

min_freq = 3 # 단어가 최소한 몇 번(3번)이상 나와야 사전에 포함시킴 (threshold)

vocab_words = [w for w, c in word_counter.items() if c >= min_freq]
len(vocab_words) # 482 >> 기준 (등장빈도 3번 이상) >> 사전에 포할된 단어 수


466

In [ ]:
idx2word = []   # index 에서 단어로 바꾸어 주는 리스트 (idx >> word)
idx2word.extend(special_tokens) # 앞쪽에 특수 토큰들을 순서대로 추가
idx2word.extend(sorted(vocab_words))  # 나머지 단어들을 정렬하여 뒤에 붙이기

word2idx = {w: i for i, w in enumerate(idx2word)}

pad_idx = word2idx['<pad>']
start_idx = word2idx['<start>']
end_idx = word2idx['<end>']
unk_idx = word2idx['<unk>']

len(idx2word)

470

In [ ]:
vocab_size = len(idx2word) # 최종 단어사전 크기 (특수 토큰 포함)
vocab_size

470

In [ ]:
# 문장을 숫자 시퀀스로 변환하는 함수
def sentence_to_indices(sentence: str, max_len: int=20):
    tokens = sentence.split()
    indices = [start_idx] # 문장 시작을 의미하는 토큰 인덱스 맨 앞에 추가
    for w in tokens:
       idx = word2idx.get(w, unk_idx) # 단어가 사전에 있으면 그 인덱스, 없으면 unk index 가져옴
       indices.append(idx)
       if len(indices) >= max_len - 1:  # 너무 길다면 (마지막에 <end> 넣어야 하는데)
          break
    indices.append(end_idx) # 문장 끝을 의미하는 토큰 인덱스 마지막에 추가
    if len(indices) < max_len:
        indices.extend([pad_idx] * (max_len-len(indices))) # 남은 부분 모두 <pad> 채움
    return indices

# 예시로 한 문장을 숫자 시퀀스로 변환해 볼까?
# captions_dict[small_image_filenames[0]][0]

example_caption = captions_dict[small_image_filenames[0]][0]
print(example_caption)
# sentence_to_indices(example_caption)
# [1, 4, 228, 186, 4, 484, 344, 3, 4, 49, 296, 2, 0, 0, 0, 0, 0, 0, 0, 0]
example_indices = sentence_to_indices(example_caption, max_len=10) # 최대 길이 10으로 제한
print(example_indices)

there are two dogs in the snow and one has something in his mouth
[1, 411, 18, 436, 121, 198, 408, 369, 15, 2]


In [ ]:
# example_indices 를 다시 단어로 변경
[idx2word[i] for i in example_indices]

['<start>', 'there', 'are', 'two', 'dogs', 'in', 'the', 'snow', 'and', '<end>']